# 03 · MemoryOS — 실제 대화로 동작 확인

MemoryOS에 실제 LoCoMo 대화를 넣고, 세그먼트 생성 → 검색 → 승격 →
eviction이 실제로 어떻게 일어나는지 단계별로 확인한다.

```
STM (큐 7칸)          최근 대화 + chain 요약
  ↓ 차면 FIFO 이관
MTM (세그먼트)        주제별 묶음, heat = 0.8·N_visit + 0.8·L + γ·recency
  ↓ heat > 5 승격        ↓ 자리 부족하면 heat 최저 삭제
LPM (영구)            사용자 프로필 + 사실들
```

**데이터**: conv-26 세션 3·4. S3(6월 9일)는 학교 연설 이벤트 이야기,
S4(6월 27일)는 목걸이·근황. eviction을 관찰하기 위해 MTM을 **4칸**으로
줄인다 (논문 기본값 200 — 데모용 설정).

> 이 노트북은 실제 LLM(Groq)을 호출한다: **~110호출, ~10분, GROQ_API_KEY 필요.**

In [1]:
from memlab.data import load_locomo
from memlab.evaluation import set_f1
from memlab.llm import GroqProvider
from memlab.methods import Utterance
from memlab.methods.memoryos import MemoryOS, MemoryOSConfig

sample = load_locomo()[0]
s3 = next(s for s in sample.sessions if s.index == 3)
s4 = next(s for s in sample.sessions if s.index == 4)

llm = GroqProvider()
victims = []  # eviction 관찰: 삭제되는 세그먼트가 여기 쌓인다
memoryos = MemoryOS(llm, sample.speaker_a, sample.speaker_b,
                    config=MemoryOSConfig(mtm_capacity=4),
                    on_evict=victims.append)

def ingest_session(session):
    for turn in session.turns:
        memoryos.ingest(Utterance(turn.speaker, turn.text,
                                 session.date_time, turn.blip_caption))

def memory_map(title):
    print(f"── {title} ──")
    for seg in sorted(memoryos.mtm.segments.values(), key=lambda g: -g.H_segment):
        print(f"  {len(seg.details)}p N={seg.N_visit} heat={seg.H_segment:.1f}"
              f"  {seg.summary[:58]}")

print(f"{sample.speaker_a} & {sample.speaker_b} / "
      f"S3 {len(s3.turns)}발화({s3.date_time}) + S4 {len(s4.turns)}발화({s4.date_time})")

Caroline & Melanie / S3 23발화(7:55 pm on 9 June, 2023) + S4 18발화(10:37 am on 27 June, 2023)


## 1. Ingest — 대화가 세그먼트로 묶인다

S3을 발화 순서대로 넣는다. 페이지마다 LLM이 chain을 판단·요약하고,
STM에서 밀려난 페이지는 MTM 세그먼트로 묶인다.

In [2]:
ingest_session(s3)
memoryos.flush_open_page()

memory_map("S3 ingest 후 MTM — 세그먼트별 (크기, 방문, heat, LLM이 쓴 요약)")

sample_page = memoryos.stm.get_all()[-1]
print(f"\nchain의 산물 — 한 페이지의 meta_chain (LLM이 쓴 실물):")
print(f"  {sample_page.meta_info}")
print(f"\n지금까지 {llm.calls}호출 / {llm.total_tokens:,}토큰")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

── S3 ingest 후 MTM — 세그먼트별 (크기, 방문, heat, LLM이 쓴 요약) ──
  1p N=0 heat=0.8  The conversation highlights the importance of sharing pers
  1p N=0 heat=0.8  Caroline expresses her commitment to using her voice to fo
  1p N=0 heat=0.8  The conversation focuses on mutual encouragement and share
  1p N=0 heat=0.8  The user expresses gratitude for the strong support and mo

chain의 산물 — 한 페이지의 meta_chain (LLM이 쓴 실물):
  Caroline shares her experience speaking at a school event about her transgender journey, receiving encouragement from Melanie. The conversation then shifts to personal life updates, where both friends discuss their support systems and share photos of their families and wedding.

지금까지 41호출 / 16,146토큰


## 2. 검색 — 검색된 세그먼트는 heat가 오른다

논문 3.3: 검색에 걸린 세그먼트는 N_visit이 올라간다.
연설 이벤트에 대해 다섯 번 질문하고 heat 변화를 확인한다.

In [3]:
rehearsal_questions = [
    "What event did Caroline speak at?",
    "How did Caroline feel about her school speech?",
    "Who supported Caroline at the event?",
    "What did Caroline talk about at the school event?",
    "What community was Caroline's speech about?",
]
for question in rehearsal_questions:
    answer = memoryos.answer(question)
    print(f"Q: {question}\nA: {answer}\n")

memory_map("검색 5회 후 — heat 변화")

Q: What event did Caroline speak at?
A: school event



Q: How did Caroline feel about her school speech?
A: grateful



Q: Who supported Caroline at the event?
A: friends, family and mentors



Q: What did Caroline talk about at the school event?
A: her transgender journey



Q: What community was Caroline's speech about?
A: transgender

── 검색 5회 후 — heat 변화 ──
  1p N=5 heat=4.8  The conversation highlights the importance of sharing pers
  1p N=5 heat=4.8  Caroline expresses her commitment to using her voice to fo
  1p N=5 heat=4.8  The conversation focuses on mutual encouragement and share
  1p N=5 heat=4.8  The user expresses gratitude for the strong support and mo


## 3. 승격 — heat > 5 세그먼트는 LPM으로 넘어간다

S4를 이어서 넣으면 페이지가 처리될 때마다 승격 조건을 검사한다.
2단계에서 heat가 오른 세그먼트가 문턱을 넘는지 확인한다.

In [4]:
kb_size_before = len(memoryos.lpm.user_kb)
ingest_session(s4)
memoryos.flush_open_page()

print(f"승격 발생: KB {kb_size_before}개 → {len(memoryos.lpm.user_kb)}개\n")
print("── LPM: User Profile (앞부분) ──")
print(memoryos.lpm.user_profile[:400])
print("\n── LPM: User KB (LLM이 추출한 사실들) ──")
for fact in memoryos.lpm.user_kb.all_texts():
    print(f"  - {fact}")

승격 발생: KB 0개 → 2개

── LPM: User Profile (앞부분) ──
Need for Belonging (High): The user explicitly states that friends, family, and mentors are their "rocks" and source of strength, emphasizing the critical role of their support system.
Family Concern (High): The user shares a photo of their family and discusses the importance of family and friends in their life, indicating a strong focus on familial and close social bonds.
Emotional Expression (Hi

── LPM: User KB (LLM이 추출한 사실들) ──
  - Moved from home country 4 years ago
  - Experienced breakup


## 4. Eviction — 어떤 세그먼트가 지워지는가

MTM이 4칸뿐이라 S4가 흐르는 동안 heat가 가장 낮은 세그먼트가 계속
지워졌다. 무엇이 지워졌는지 보면 이 시스템의 삭제 정책이 드러난다.

In [5]:
print(f"삭제된 세그먼트 {len(victims)}개 — (크기, 방문수, 내용):")
for victim in victims:
    print(f"  {len(victim.details)}p N={victim.N_visit}  {victim.summary[:55]}")

memory_map("\n생존자")

삭제된 세그먼트 10개 — (크기, 방문수, 내용):
  1p N=0  Caroline shares her positive experience leading a schoo
  1p N=0  Caroline shares the empowering experience of delivering
  1p N=0  The user compliments a wedding photo and asks about the
  1p N=0  Caroline congratulates Melanie on her wedding, and Mela
  1p N=0  A joyful family day spent playing games, sharing meals,
  1p N=0  Cherishing family moments
  1p N=0  The value of family and spending time with loved ones.
  1p N=0  Caroline shares a photo of a new necklace featuring a c
  1p N=0  Caroline shares the sentimental significance of a neckl
  1p N=0  The conversation covers Caroline sharing a sentimental 
── 
생존자 ──
  1p N=5 heat=4.8  The conversation highlights the importance of sharing pers
  1p N=5 heat=4.8  Caroline expresses her commitment to using her voice to fo
  1p N=5 heat=4.8  The conversation focuses on mutual encouragement and share
  2p N=5 heat=4.0  The conversation focuses on the importance of personal sup


지워진 것은 전부 **작고(1페이지) 한 번도 검색되지 않은 새 세그먼트**다.
heat = 0.8·N + 0.8·L에서 recency 항(γ=0.0001)이 사실상 꺼져 있어서,
새 세그먼트는 자리 잡기 전에 항상 최저 heat다.
**무엇을 왜 지우는지 묻지 않는 삭제** — 이 문제가 이후 우리 연구
(cause-aware forgetting)의 출발점이다.

## 5. QA — 기억의 상태가 점수를 가른다

상태가 다른 다섯 문제를 골라 답하게 한다:
검색으로 살아남은 세그먼트 / 두 세션 종합(multi-hop) / 지워진 세그먼트 /
경계 사례 / STM에 남아 있는 최근 페이지.

In [6]:
exam = [
    (8, "여러 번 검색된 연설 내용"),
    (11, "두 세션 종합 (multi-hop)"),
    (90, "삭제된 결혼 이야기"),
    (95, "경계 사례 — S4 중반의 캠핑"),
    (100, "STM에 남은 S4 마지막 화제"),
]
for index, fate in exam:
    qa = sample.qa[index]
    prediction = memoryos.answer(qa.question)
    print(f"[{fate}]")
    print(f"Q: {qa.question}")
    print(f"  정답: {qa.answer}")
    print(f"  예측: {prediction}")
    print(f"  set_f1 = {set_f1(prediction, qa.answer):.2f}\n")

print(f"총 비용: {llm.calls}호출 / {llm.total_tokens:,}토큰")

[여러 번 검색된 연설 내용]
Q: When did Caroline give a speech at a school?
  정답: The week before 9 June 2023
  예측: June 2023
  set_f1 = 0.50



[두 세션 종합 (multi-hop)]
Q: Where did Caroline move from 4 years ago?
  정답: Sweden
  예측: home country
  set_f1 = 0.00



[삭제된 결혼 이야기]
Q: How long have Mel and her husband been married?
  정답: Mel and her husband have been married for 5 years.
  예측: Information not provided
  set_f1 = 0.00



[경계 사례 — S4 중반의 캠핑]
Q: What did Melanie and her family do while camping?
  정답: explored nature, roasted marshmallows, and went on a hike
  예측: explored nature, roasted marshmallows around the campfire and went on a hike
  set_f1 = 0.86



[STM에 남은 S4 마지막 화제]
Q: What kind of place does Caroline want to create for people?
  정답: a safe and inviting place for people to grow
  예측: a safe, inviting place for people to grow
  set_f1 = 0.94

총 비용: 108호출 / 48,874토큰


점수가 기억의 상태를 그대로 따라간다 — 그리고 multi-hop 실패(0.00)의
원인이 흥미롭다. KB에 승격된 사실을 보면 *"Moved from **home country**
4 years ago"* — 지식 추출 과정에서 "Sweden"이 **일반적인 표현으로
뭉개졌다**. 원본 페이지(목걸이 이야기)는 이미 지워졌으니, 남은 건
뭉개진 요약뿐.

**eviction은 원본을 지우고, 요약은 세부를 지운다.** 둘이 겹치면 답할 수 없다.

## 정리

- **Ingest**: 대화가 chain 요약을 달고 주제별 세그먼트로 묶인다 (실제론 잘게 쪼개진다)
- **검색 → 승격**: 자주 검색된 세그먼트는 heat가 올라 LPM으로 넘어간다
- **Eviction**: 자리가 부족하면 heat가 가장 낮은 세그먼트가 지워진다 — 지워지는 건
  늘 작고 새로운 세그먼트다 (recency 항이 꺼져 있어서). 무엇을 왜 지우는지 묻지 않는다
- QA 점수는 기억의 상태를 그대로 따라간다

**다음 — 04 · 실험**: 대화 10편 전체를 돌려 baseline 수치를 얻는다
(러너, 체크포인트, 무료 티어 한도 관리).